In [0]:
%sql
select last_run_status,process_group, * from sandbox.migration_config.table_migration_config 
where 1=1
-- -- and process_group = 'TPCH_autoloader' 
-- order by process_group, src_table


-- update sandbox.migration_config.table_migration_config set 
-- incremental_col_type = 
-- merge_keys = 'n_nationkey' 

-- UPDATE sandbox.migration_config.table_migration_config
-- SET
    -- source_type     = 'volume',
    -- src_database    = 'sandbox',
    -- src_schema      = 'migration_config_bronze',
    -- last_run_status = 'PENDING',
    -- notes           = NULL

In [0]:
%sql
-- SELECT *
-- FROM sandbox.migration_config.table_migration_config
-- WHERE process_group = 'TPCH_autoloader';

UPDATE sandbox.migration_config.table_migration_config
SET
    source_type     = 'volume',
    src_database    = 'sandbox',
    src_schema      = 'migration_config_bronze',
    last_run_status = 'PENDING',
    notes           = NULL
WHERE process_group = 'TPCH_autoloader';

In [0]:
%sql
select * from sandbox.migration_config.source_connection_config
WHERE connection_name = 'pg_neon';

-- UPDATE sandbox.migration_config.source_connection_config
-- SET connection_method  = 'jdbc',
--     catalog_name       = NULL,
--     jdbc_url_template  = 'jdbc:postgresql://{host}:{port}/{database}?sslmode=require',
--     driver_class       = 'org.postgresql.Driver'
-- WHERE connection_name = 'pg_neon';

In [0]:
%sql

UPDATE sandbox.migration_config.table_migration_config
SET src_database   = 'file',
    src_schema  = 'tpch'
where process_group = 'TPCH_autoloader'

In [0]:
%sql
-- TRUNCATE TABLE sandbox.migration_config_bronze.tpch_supplier_autoloader
TRUNCATE TABLE sandbox.migration_config_bronze.tpch_nation_autoloader

In [0]:
dbutils.fs.rm(
    "/Volumes/sandbox/migration_config_bronze/landing/_checkpoints/TPCH_autoloader/supplier/",
    recurse=True
)
dbutils.fs.rm(
    "/Volumes/sandbox/migration_config_bronze/landing/_checkpoints/TPCH_autoloader/nation/",
    recurse=True
)
# Also delete the schema checkpoint
dbutils.fs.rm(
    "/Volumes/sandbox/migration_config_bronze/landing/_checkpoints/TPCH_autoloader/supplier_schema/",
    recurse=True
)
dbutils.fs.rm(
    "/Volumes/sandbox/migration_config_bronze/landing/_checkpoints/TPCH_autoloader/nation_schema/",
    recurse=True
)
print("Checkpoints cleared")

In [0]:
%sql
UPDATE sandbox.migration_config.table_migration_config
SET last_run_status   = 'PENDING',
    last_loaded_value  = NULL,
    last_sf_row_count  = NULL,
    last_delta_count   = NULL,
    notes              = NULL
where process_group = 'TPCH_autoloader'

In [0]:
%sql
-- SELECT 1 FROM pg_neon.public.supplier_pg LIMIT 1

SELECT table_name, table_schema
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;

In [0]:
import requests

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point \
    .getDbutils().notebook().getContext() \
    .apiToken().get()

headers = {"Authorization": f"Bearer {token}"}

r = requests.post(
    f"https://{workspace_url}/api/2.0/secrets/put",
    headers=headers,
    json={
        "scope"        : "pg_neon",
        "key"          : "password",
        "string_value" : "Prai$3him!"
    }
)
print(r.status_code, r.json())

In [0]:
host     = dbutils.secrets.get("pg_neon", "host")
port     = dbutils.secrets.get("pg_neon", "port")
user     = dbutils.secrets.get("pg_neon", "user")
password = dbutils.secrets.get("pg_neon", "password")

print(f"Host: {host}")
print(f"Port: {port}")
print(f"User: {user}")
print(f"Password set: {len(password) > 0}")

jdbc_url = f"jdbc:postgresql://{host}:{port}/neondb?sslmode=require"

df = spark.read.format("jdbc") \
    .option("url",      jdbc_url) \
    .option("dbtable",  "(SELECT 1 AS test) AS t") \
    .option("user",     user) \
    .option("password", password) \
    .option("driver",   "org.postgresql.Driver") \
    .load()

df.show()
print("✅ Connected to Neon successfully")

In [0]:
import requests

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point \
    .getDbutils().notebook().getContext() \
    .apiToken().get()

headers = {"Authorization": f"Bearer {token}"}

# List the secret to confirm it was updated (shows timestamp not value)
r = requests.get(
    f"https://{workspace_url}/api/2.0/secrets/list",
    headers=headers,
    params={"scope": "pg_neon"}
)
for s in r.json().get('secrets', []):
    print(f"{s['key']}: last updated {s['last_updated_timestamp']}")

In [0]:
import requests

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point \
    .getDbutils().notebook().getContext() \
    .apiToken().get()

headers = {"Authorization": f"Bearer {token}"}
base_url = f"https://{workspace_url}/api/2.0/secrets/put"

updates = {
    "host"    : "ep-summer-sun-ajvmjk1o-pooler.c-3.us-east-2.aws.neon.tech",
    "password": "npg_UstoH40KPWNM"
}

for key, value in updates.items():
    r = requests.post(base_url, headers=headers,
        json={"scope": "pg_neon", "key": key, "string_value": value})
    print(f"{key}: {r.status_code}")

In [0]:
%sql
UPDATE sandbox.migration_config.table_migration_config
SET last_run_status = 'PENDING',
    notes           = NULL
WHERE table_id = 'pg_neon_public_supplier_pg_001';